# 🤖 AI Agent Programming Guide

## A Hands-On Guide to Building AI Agents with Ollama & Python

---

### 📋 What You'll Build Today

| Part | Topic | What You'll Learn |
|------|-------|-------------------|
| 1️⃣ | Understanding Agents | Agent vs LLM, core components |
| 2️⃣ | Setting Up Ollama | Install, pull models, Python integration |
| 3️⃣ | Your First Agent | ReAct pattern, tool calling |
| 4️⃣ | Adding Memory | Conversation history, context management |
| 5️⃣ | Multi-Tool Agent | Multiple tools working together |
| 6️⃣ | Self-Reflection | Chain-of-thought, self-critique |
| 7️⃣ | Complete Project | Research Assistant Agent |

---

### 🎯 Prerequisites

- Python 3.8+
- Ollama installed ([ollama.ai](https://ollama.ai))
- At least one model pulled: `ollama pull llama3.2` or `ollama pull mistral`

---

# 📚 Part 1: Understanding AI Agents

## What Makes an Agent Different from an LLM?

```
┌────────────────────────────────────────────────────────────────────────┐
│                    SIMPLE LLM vs AI AGENT                              │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  SIMPLE LLM:                                                           │
│  ┌─────────┐          ┌─────────┐                                      │
│  │  Input  │ ───────▶ │ Output  │   (One-shot: in → out)               │
│  └─────────┘          └─────────┘                                      │
│                                                                        │
│  AI AGENT:                                                             │
│  ┌─────────┐   ┌─────────────────────────────────────────┐             │
│  │  Goal   │ → │  LOOP: Think → Act → Observe → Repeat  │ → │ Result │ │
│  └─────────┘   └─────────────────────────────────────────┘             │
│                         ↓                                              │
│                   Uses TOOLS 🔧                                        │
│                   Has MEMORY 🧠                                        │
│                   Makes DECISIONS 💭                                   │
└────────────────────────────────────────────────────────────────────────┘
```

## The Agent Loop (Perceive → Think → Act → Observe)

```
                    ┌────────────────────────────────────┐
                    │         THE AGENT LOOP             │
                    └────────────────────────────────────┘
                                    │
          ┌─────────────────────────┴─────────────────────────┐
          ▼                                                   │
    ┌──────────┐                                              │
    │ PERCEIVE │  ← User input + Current state + Memory       │
    └────┬─────┘                                              │
         │                                                    │
         ▼                                                    │
    ┌──────────┐                                              │
    │  THINK   │  ← LLM reasons: "What should I do next?"     │
    └────┬─────┘                                              │
         │                                                    │
         ▼                                                    │
    ┌──────────┐                                              │
    │   ACT    │  ← Execute a tool OR respond to user         │
    └────┬─────┘                                              │
         │                                                    │
         ▼                                                    │
    ┌──────────┐                                              │
    │ OBSERVE  │  ← Get tool result, update memory            │
    └────┬─────┘                                              │
         │                                                    │
         └────────────────────────────────────────────────────┘
                    (Repeat until task complete)
```

## Core Components of an Agent

| Component | Role | Example |
|-----------|------|--------|
| 🧠 **Brain (LLM)** | Reasoning, decision making | Llama3.2, Mistral, GPT-4 |
| 🔧 **Tools** | Actions the agent can take | Calculator, web search, file I/O |
| 📝 **Memory** | Context from past interactions | Chat history, user preferences |
| 📋 **Planning** | Breaking tasks into steps | Task decomposition, reflection |

---

# 🔧 Part 2: Setting Up Ollama

## Step 1: Install Ollama

**Windows:** Download from [ollama.ai](https://ollama.ai) and run installer

**Mac/Linux:**
```bash
curl -fsSL https://ollama.ai/install.sh | sh
```

## Step 2: Pull a Model

```bash
# Recommended for agents (7B parameter model, good reasoning)
ollama pull llama3.2

# Alternative: faster, smaller
ollama pull mistral

# Alternative: very small but capable
ollama pull phi3
```

## Step 3: Verify It's Working

```bash
# Test in terminal
ollama run llama3.2 "Hello, can you see me?"
```

In [ ]:
# 📦 Install Python Dependencies
!pip install ollama

In [ ]:
# 🔗 Connect to Ollama
import ollama

# Check connection and available models
print("🔍 Checking Ollama connection...")
try:
    models = ollama.list()
    print("✅ Connected to Ollama!")
    print("\n📋 Available models:")
    for model in models['models']:
        print(f"   • {model['name']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\n💡 Make sure Ollama is running:")
    print("   1. Open a terminal")
    print("   2. Run: ollama serve")

In [ ]:
# 💬 Basic Chat with Ollama
MODEL = "llama3.2"  # Change this to your pulled model

def chat(prompt: str) -> str:
    """Simple chat with the LLM."""
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response['message']['content']

# Test it!
print("🤖 Testing basic chat...")
response = chat("What is 2 + 2? Answer in one word.")
print(f"Response: {response}")

---

# 🛠️ Part 3: Building Your First Agent (ReAct Pattern)

## What is ReAct?

**ReAct** = **Re**asoning + **Act**ing

The LLM explicitly states its reasoning before taking actions:

```
User: What's the weather in Tokyo and should I bring an umbrella?

Agent:
THOUGHT: I need to check the weather in Tokyo first.
ACTION: get_weather("Tokyo")
OBSERVATION: Temperature: 15°C, Condition: Rainy

THOUGHT: It's rainy, so the user should bring an umbrella.
ACTION: respond("It's 15°C and rainy in Tokyo. Yes, bring an umbrella!")
```

## The Pattern

```
THOUGHT: [What I'm thinking]
ACTION: tool_name(arguments)
OBSERVATION: [Result from tool]
... repeat until done ...
THOUGHT: I have enough information to answer.
ACTION: respond(final_answer)
```

In [ ]:
# 🔧 Step 1: Define Our Tools

import json
import math

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        # Safe evaluation of math expressions
        allowed_names = {
            'abs': abs, 'round': round,
            'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
            'sqrt': math.sqrt, 'pow': pow, 'log': math.log,
            'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

def get_weather(city: str) -> str:
    """Get weather for a city (simulated for demo)."""
    # In real app, call a weather API
    weather_data = {
        "tokyo": {"temp": 15, "condition": "Rainy", "humidity": 80},
        "london": {"temp": 12, "condition": "Cloudy", "humidity": 75},
        "new york": {"temp": 22, "condition": "Sunny", "humidity": 45},
        "mumbai": {"temp": 32, "condition": "Humid", "humidity": 85},
        "chennai": {"temp": 34, "condition": "Partly Cloudy", "humidity": 70},
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        w = weather_data[city_lower]
        return f"Weather in {city}: {w['temp']}°C, {w['condition']}, Humidity: {w['humidity']}%"
    return f"Weather data not available for {city}"

def get_time(timezone: str) -> str:
    """Get current time (simulated for demo)."""
    from datetime import datetime
    # Simplified timezone handling
    offsets = {
        "ist": 5.5, "jst": 9, "gmt": 0, "est": -5, "pst": -8
    }
    tz = timezone.lower()
    if tz in offsets:
        import time
        # Get UTC and add offset
        utc_hour = datetime.utcnow().hour
        local_hour = (utc_hour + offsets[tz]) % 24
        return f"Current time ({timezone.upper()}): {int(local_hour):02d}:{datetime.utcnow().minute:02d}"
    return f"Unknown timezone: {timezone}"

# Define tool registry
TOOLS = {
    "calculator": {
        "function": calculator,
        "description": "Evaluate mathematical expressions. Usage: calculator(expression)"
    },
    "get_weather": {
        "function": get_weather,
        "description": "Get weather for a city. Usage: get_weather(city_name)"
    },
    "get_time": {
        "function": get_time,
        "description": "Get current time in a timezone. Usage: get_time(timezone) e.g., IST, JST, GMT, EST, PST"
    }
}

print("🔧 Tools registered:")
for name, info in TOOLS.items():
    print(f"   • {name}: {info['description']}")

In [ ]:
# 🧠 Step 2: Create the Agent System Prompt

def get_system_prompt():
    """Generate the system prompt with available tools."""
    tools_desc = "\n".join([
        f"  - {name}: {info['description']}" 
        for name, info in TOOLS.items()
    ])
    
    return f"""You are a helpful AI agent that can use tools to accomplish tasks.

AVAILABLE TOOLS:
{tools_desc}
  - respond: Give final answer to user. Usage: respond(your_answer)

INSTRUCTIONS:
1. Always think step-by-step before acting
2. Use this EXACT format for every response:

THOUGHT: [Your reasoning about what to do next]
ACTION: tool_name(argument)

3. After seeing an OBSERVATION, continue with another THOUGHT and ACTION
4. When you have the final answer, use: ACTION: respond(your_final_answer)
5. NEVER skip the THOUGHT step
6. Use ONLY the tools listed above

EXAMPLE:
User: What is 25 * 4?

THOUGHT: I need to calculate 25 * 4. I'll use the calculator tool.
ACTION: calculator(25 * 4)
"""

print("📋 System Prompt:")
print("=" * 50)
print(get_system_prompt())
print("=" * 50)

In [ ]:
# 🔍 Step 3: Parse Agent Response to Extract Actions

import re

def parse_agent_response(response: str) -> dict:
    """Parse the agent's response to extract thought and action."""
    result = {
        "thought": None,
        "action": None,
        "action_name": None,
        "action_arg": None
    }
    
    # Extract THOUGHT
    thought_match = re.search(r'THOUGHT:\s*(.+?)(?=ACTION:|$)', response, re.DOTALL | re.IGNORECASE)
    if thought_match:
        result["thought"] = thought_match.group(1).strip()
    
    # Extract ACTION
    action_match = re.search(r'ACTION:\s*(.+?)(?=\n|$)', response, re.IGNORECASE)
    if action_match:
        result["action"] = action_match.group(1).strip()
        
        # Parse action name and arguments
        # Match patterns like: tool_name(argument) or tool_name("argument")
        tool_match = re.match(r'(\w+)\((.*)\)', result["action"])
        if tool_match:
            result["action_name"] = tool_match.group(1)
            # Clean up the argument (remove quotes if present)
            arg = tool_match.group(2).strip()
            arg = arg.strip('"').strip("'")
            result["action_arg"] = arg
    
    return result

# Test parsing
test_response = """THOUGHT: I need to check the weather in Tokyo.
ACTION: get_weather("Tokyo")"""

parsed = parse_agent_response(test_response)
print("🔍 Test Parsing:")
print(f"   Thought: {parsed['thought']}")
print(f"   Action: {parsed['action']}")
print(f"   Tool: {parsed['action_name']}")
print(f"   Argument: {parsed['action_arg']}")

In [ ]:
# 🤖 Step 4: The Agent Loop!

def run_agent(user_query: str, max_iterations: int = 5, verbose: bool = True) -> str:
    """
    Run the agent to answer a user query.
    
    Args:
        user_query: The user's question or task
        max_iterations: Maximum number of think-act cycles
        verbose: Whether to print intermediate steps
    
    Returns:
        The agent's final response
    """
    # Initialize conversation with system prompt
    messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": user_query}
    ]
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"🎯 USER QUERY: {user_query}")
        print(f"{'='*60}\n")
    
    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1} ---")
        
        # Get agent response
        response = ollama.chat(model=MODEL, messages=messages)
        agent_response = response['message']['content']
        
        if verbose:
            print(f"\n🤖 AGENT:")
            print(agent_response)
        
        # Parse the response
        parsed = parse_agent_response(agent_response)
        
        if not parsed['action_name']:
            if verbose:
                print("\n⚠️ No valid action found. Ending.")
            return agent_response
        
        # Check if agent wants to respond (final answer)
        if parsed['action_name'].lower() == 'respond':
            if verbose:
                print(f"\n✅ FINAL ANSWER: {parsed['action_arg']}")
            return parsed['action_arg']
        
        # Execute the tool
        if parsed['action_name'] in TOOLS:
            tool_func = TOOLS[parsed['action_name']]['function']
            observation = tool_func(parsed['action_arg'])
            
            if verbose:
                print(f"\n🔧 TOOL EXECUTED: {parsed['action_name']}({parsed['action_arg']})")
                print(f"📊 OBSERVATION: {observation}")
            
            # Add agent response and observation to messages
            messages.append({"role": "assistant", "content": agent_response})
            messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
        else:
            if verbose:
                print(f"\n❌ Unknown tool: {parsed['action_name']}")
            messages.append({"role": "assistant", "content": agent_response})
            messages.append({"role": "user", "content": f"OBSERVATION: Error - Unknown tool '{parsed['action_name']}'. Use only: {list(TOOLS.keys())}"})
    
    return "Agent reached maximum iterations without a final answer."

print("✅ Agent function created!")
print("🎯 Ready to run queries!")

In [ ]:
# 🎮 Test the Agent!

# Test 1: Simple calculation
result = run_agent("What is 15 * 7 + 23?")

In [ ]:
# Test 2: Weather query
result = run_agent("What's the weather like in Mumbai?")

In [ ]:
# Test 3: Multi-step reasoning
result = run_agent("Is it hotter in Mumbai or Tokyo right now? By how many degrees?")

---

# 🧠 Part 4: Adding Memory to Your Agent

## Why Memory Matters

Without memory, the agent forgets everything after each query.

```
WITHOUT MEMORY:                        WITH MEMORY:
─────────────────────                  ─────────────────────
User: My name is Aarav                 User: My name is Aarav
Bot: Nice to meet you, Aarav!          Bot: Nice to meet you, Aarav!

User: What's my name?                  User: What's my name?
Bot: I don't know your name.  ❌       Bot: Your name is Aarav!  ✅
```

## Types of Memory

| Type | Duration | Example |
|------|----------|--------|
| **Short-term** | Current conversation | Chat history |
| **Long-term** | Persists across sessions | User preferences, facts |

In [ ]:
# 🧠 Agent with Conversation Memory

class MemoryAgent:
    """
    An agent that remembers conversation history.
    """
    
    def __init__(self, model: str = MODEL, max_history: int = 10):
        self.model = model
        self.max_history = max_history
        self.conversation_history = []
        self.system_prompt = get_system_prompt()
    
    def _trim_history(self):
        """Keep only the last N exchanges to prevent context overflow."""
        if len(self.conversation_history) > self.max_history * 2:
            self.conversation_history = self.conversation_history[-self.max_history * 2:]
    
    def chat(self, user_message: str, max_iterations: int = 5, verbose: bool = True) -> str:
        """Chat with the agent, maintaining conversation history."""
        
        # Add user message to history
        self.conversation_history.append({"role": "user", "content": user_message})
        
        # Build full messages with system prompt + history
        messages = [
            {"role": "system", "content": self.system_prompt}
        ] + self.conversation_history
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"🎯 USER: {user_message}")
            print(f"📜 History: {len(self.conversation_history)} messages")
            print(f"{'='*60}")
        
        # Agent loop (similar to before)
        working_messages = messages.copy()
        
        for iteration in range(max_iterations):
            response = ollama.chat(model=self.model, messages=working_messages)
            agent_response = response['message']['content']
            
            if verbose:
                print(f"\n🤖 AGENT (iter {iteration + 1}):")
                print(agent_response)
            
            parsed = parse_agent_response(agent_response)
            
            if not parsed['action_name']:
                # No action - treat as direct response
                self.conversation_history.append({"role": "assistant", "content": agent_response})
                self._trim_history()
                return agent_response
            
            if parsed['action_name'].lower() == 'respond':
                final_answer = parsed['action_arg']
                self.conversation_history.append({"role": "assistant", "content": final_answer})
                self._trim_history()
                if verbose:
                    print(f"\n✅ FINAL: {final_answer}")
                return final_answer
            
            # Execute tool
            if parsed['action_name'] in TOOLS:
                tool_func = TOOLS[parsed['action_name']]['function']
                observation = tool_func(parsed['action_arg'])
                
                if verbose:
                    print(f"\n🔧 {parsed['action_name']}({parsed['action_arg']}) → {observation}")
                
                working_messages.append({"role": "assistant", "content": agent_response})
                working_messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
            else:
                working_messages.append({"role": "assistant", "content": agent_response})
                working_messages.append({"role": "user", "content": f"OBSERVATION: Unknown tool '{parsed['action_name']}'"})
        
        return "Max iterations reached."
    
    def clear_memory(self):
        """Clear conversation history."""
        self.conversation_history = []
        print("🧹 Memory cleared!")
    
    def show_memory(self):
        """Display current conversation history."""
        print("\n📜 CONVERSATION HISTORY:")
        print("-" * 40)
        for msg in self.conversation_history:
            role = "👤 User" if msg['role'] == 'user' else "🤖 Agent"
            content = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
            print(f"{role}: {content}")
        print("-" * 40)

# Create the memory agent
agent = MemoryAgent()
print("✅ Memory Agent created!")

In [ ]:
# 🧪 Test Memory Across Multiple Turns

# Turn 1: Introduce yourself
agent.chat("Hi! My name is Priya and I'm from Chennai.", verbose=True)

In [ ]:
# Turn 2: Ask about weather in home city
agent.chat("What's the weather like in my home city?", verbose=True)

In [ ]:
# Turn 3: Test if it remembers your name
agent.chat("Do you remember my name?", verbose=True)

In [ ]:
# View the memory
agent.show_memory()

---

# 🔄 Part 5: Multi-Tool Agent

Let's add more tools and see how the agent combines them!

In [ ]:
# 🔧 Add More Tools

def search_web(query: str) -> str:
    """Simulate web search (mock data for demo)."""
    # In real app, use an actual search API
    mock_results = {
        "python": "Python is a high-level programming language known for its readability. Created by Guido van Rossum in 1991.",
        "ai": "Artificial Intelligence (AI) is the simulation of human intelligence by machines. Key areas include ML, NLP, and computer vision.",
        "ollama": "Ollama is a tool to run large language models locally. It supports models like Llama, Mistral, and Phi.",
        "agent": "AI Agents are autonomous systems that can perceive, reason, and act to achieve goals using tools and memory.",
    }
    query_lower = query.lower()
    for key, result in mock_results.items():
        if key in query_lower:
            return f"Search result for '{query}': {result}"
    return f"Search result for '{query}': No specific results found. Try a more specific query."

def translate(text_and_lang: str) -> str:
    """Translate text (simplified mock for demo)."""
    # Format: "text|target_language"
    parts = text_and_lang.split("|")
    if len(parts) != 2:
        return "Error: Use format 'text|language' e.g., 'Hello|Spanish'"
    
    text, lang = parts[0].strip(), parts[1].strip().lower()
    
    translations = {
        "spanish": {"hello": "Hola", "goodbye": "Adiós", "thank you": "Gracias"},
        "french": {"hello": "Bonjour", "goodbye": "Au revoir", "thank you": "Merci"},
        "hindi": {"hello": "नमस्ते", "goodbye": "अलविदा", "thank you": "धन्यवाद"},
        "tamil": {"hello": "வணக்கம்", "goodbye": "பிரியா விடை", "thank you": "நன்றி"},
    }
    
    if lang in translations:
        text_lower = text.lower()
        if text_lower in translations[lang]:
            return f"'{text}' in {lang.title()}: {translations[lang][text_lower]}"
        return f"Translation of '{text}' to {lang.title()}: [Demo: actual API would translate here]"
    return f"Language '{lang}' not supported. Available: Spanish, French, Hindi, Tamil"

def unit_convert(conversion: str) -> str:
    """Convert units (simplified)."""
    # Format: "value unit1 to unit2" e.g., "100 km to miles"
    conversions = {
        ("km", "miles"): 0.621371,
        ("miles", "km"): 1.60934,
        ("celsius", "fahrenheit"): lambda c: c * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda f: (f - 32) * 5/9,
        ("kg", "pounds"): 2.20462,
        ("pounds", "kg"): 0.453592,
    }
    
    try:
        parts = conversion.lower().replace("to", " ").split()
        value = float(parts[0])
        from_unit = parts[1]
        to_unit = parts[2] if len(parts) > 2 else parts[-1]
        
        key = (from_unit, to_unit)
        if key in conversions:
            factor = conversions[key]
            if callable(factor):
                result = factor(value)
            else:
                result = value * factor
            return f"{value} {from_unit} = {result:.2f} {to_unit}"
        return f"Conversion from {from_unit} to {to_unit} not supported."
    except:
        return "Error: Use format 'value unit1 to unit2' e.g., '100 km to miles'"

# Add new tools to registry
TOOLS["search_web"] = {
    "function": search_web,
    "description": "Search the web for information. Usage: search_web(query)"
}
TOOLS["translate"] = {
    "function": translate,
    "description": "Translate text. Usage: translate(text|target_language) e.g., translate(Hello|Spanish)"
}
TOOLS["unit_convert"] = {
    "function": unit_convert,
    "description": "Convert units. Usage: unit_convert(value unit1 to unit2) e.g., unit_convert(100 km to miles)"
}

print("✅ New tools added!")
print("\n🔧 All available tools:")
for name in TOOLS:
    print(f"   • {name}")

In [ ]:
# Create a new agent with all tools
multi_agent = MemoryAgent()

# Test multi-tool query
multi_agent.chat(
    "I'm traveling from Mumbai to London. What's the weather difference? Also, what is 500 km in miles?",
    verbose=True
)

In [ ]:
# Test search + translation
multi_agent.chat(
    "Search for what Python is, then tell me how to say 'Thank you' in Hindi.",
    verbose=True
)

---

# 💭 Part 6: Self-Reflection & Planning

## Chain-of-Thought Prompting

Make the agent think more deeply by asking it to reason step-by-step.

## Self-Critique Pattern

The agent reviews its own answer and improves it.

In [ ]:
# 💭 Self-Reflecting Agent

def get_reflective_prompt():
    """Enhanced prompt with self-reflection."""
    tools_desc = "\n".join([
        f"  - {name}: {info['description']}" 
        for name, info in TOOLS.items()
    ])
    
    return f"""You are a thoughtful AI agent that carefully plans and reflects before acting.

AVAILABLE TOOLS:
{tools_desc}
  - respond: Give final answer. Usage: respond(your_answer)

RESPONSE FORMAT:

PLAN: [Break down the task into steps before starting]
THOUGHT: [Your reasoning for the current step]
ACTION: tool_name(argument)

After completing your task, BEFORE giving the final answer, add:

REFLECTION: [Review your work - is the answer complete and accurate?]
ACTION: respond(your_final_answer)

IMPORTANT:
- Always start with a PLAN for multi-step tasks
- Always REFLECT before giving the final answer
- If your reflection finds issues, take corrective actions
"""

class ReflectiveAgent(MemoryAgent):
    """Agent that plans and reflects on its work."""
    
    def __init__(self, model: str = MODEL):
        super().__init__(model)
        self.system_prompt = get_reflective_prompt()

# Create reflective agent
smart_agent = ReflectiveAgent()
print("✅ Reflective Agent created!")

In [ ]:
# Test with a complex task
smart_agent.chat(
    "I need to compare weather in 3 cities: Tokyo, London, and Mumbai. Tell me which is warmest and by how much.",
    max_iterations=10,
    verbose=True
)

---

# 🎯 Part 7: Complete Project - Research Assistant Agent

Let's build a complete agent that can:
- 🔍 Search for information
- 🧮 Perform calculations
- 📝 Summarize findings
- 🧠 Remember context
- 💭 Plan and reflect

In [ ]:
# 🎓 Complete Research Assistant Agent

# Add note-taking tool
notes_storage = []

def take_note(note: str) -> str:
    """Save a note for later reference."""
    notes_storage.append(note)
    return f"Note saved: '{note[:50]}...' (Total notes: {len(notes_storage)})"

def get_notes(query: str) -> str:
    """Retrieve all saved notes."""
    if not notes_storage:
        return "No notes saved yet."
    notes_text = "\n".join([f"{i+1}. {note}" for i, note in enumerate(notes_storage)])
    return f"Saved notes:\n{notes_text}"

def summarize(text: str) -> str:
    """Summarize given text (uses LLM)."""
    response = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user", 
            "content": f"Summarize this in 2-3 sentences:\n\n{text}"
        }]
    )
    return f"Summary: {response['message']['content']}"

# Add to tools
TOOLS["take_note"] = {
    "function": take_note,
    "description": "Save a note for later. Usage: take_note(note_content)"
}
TOOLS["get_notes"] = {
    "function": get_notes,
    "description": "Get all saved notes. Usage: get_notes(any)"
}
TOOLS["summarize"] = {
    "function": summarize,
    "description": "Summarize text using AI. Usage: summarize(text_to_summarize)"
}

def get_research_prompt():
    """Specialized prompt for research assistant."""
    tools_desc = "\n".join([
        f"  - {name}: {info['description']}" 
        for name, info in TOOLS.items()
    ])
    
    return f"""You are a Research Assistant AI that helps users find, analyze, and organize information.

YOUR CAPABILITIES:
{tools_desc}
  - respond: Give final answer. Usage: respond(your_answer)

YOUR APPROACH:
1. UNDERSTAND the research question
2. PLAN your research strategy
3. GATHER information using tools
4. ANALYZE and CALCULATE when needed
5. SYNTHESIZE findings into clear answers
6. SAVE important notes for context

FORMAT:
PLAN: [Your research strategy]
THOUGHT: [Current reasoning]
ACTION: tool_name(argument)

After research is complete:
SYNTHESIS: [Combine all findings]
ACTION: respond(comprehensive_answer)

Be thorough but concise. Cite your sources (tools used).
"""

class ResearchAgent(MemoryAgent):
    """Complete research assistant agent."""
    
    def __init__(self, model: str = MODEL):
        super().__init__(model)
        self.system_prompt = get_research_prompt()
    
    def clear_notes(self):
        """Clear all saved notes."""
        global notes_storage
        notes_storage = []
        print("📝 Notes cleared!")

# Create the research assistant
researcher = ResearchAgent()
print("✅ Research Assistant Agent ready!")
print("\n🔧 Available capabilities:")
for name in TOOLS:
    print(f"   • {name}")

In [ ]:
# 🎮 Demo: Research Task

researcher.chat(
    """I'm planning a trip and need help:
    1. Compare weather in Tokyo and New York
    2. Calculate: if the flight is 14 hours and costs $800, what's the cost per hour?
    3. Save a note about which city has better weather""",
    max_iterations=10,
    verbose=True
)

In [ ]:
# View saved notes
print(get_notes(""))

In [ ]:
# Follow-up using memory
researcher.chat(
    "Based on our earlier research, which city should I choose and why? Also, how do I say 'Thank you' in Japanese?",
    max_iterations=8,
    verbose=True
)

---

# 🎉 Congratulations!

## You've Built a Complete AI Agent!

### What You Learned:

| Concept | What It Does |
|---------|-------------|
| ✅ **Agent Loop** | Perceive → Think → Act → Observe |
| ✅ **ReAct Pattern** | Explicit reasoning before actions |
| ✅ **Tool Calling** | Parse LLM output, execute tools, return results |
| ✅ **Memory** | Maintain conversation context across turns |
| ✅ **Multi-Tool** | Combine multiple tools for complex tasks |
| ✅ **Self-Reflection** | Plan and review before answering |

### Next Steps:

1. 🔧 **Add real tools**: Web APIs, file operations, database queries
2. 🧠 **Persistent memory**: Store history in files/databases
3. 🔗 **Chain agents**: Multiple agents working together
4. 🛡️ **Add guardrails**: Input validation, output filtering
5. 📊 **Logging**: Track all agent actions for debugging

### Resources:

- [Ollama Documentation](https://ollama.ai)
- [ReAct Paper](https://arxiv.org/abs/2210.03629)
- [LangChain Agents](https://python.langchain.com/docs/modules/agents/)
- [AutoGPT](https://github.com/Significant-Gravitas/AutoGPT)

In [ ]:
# 🏁 Final Interactive Session

print("="*60)
print("🤖 INTERACTIVE AGENT SESSION")
print("="*60)
print("Type your questions below! The agent will use all its tools.")
print("Type 'quit' to exit, 'memory' to view history, 'clear' to reset.")
print("="*60)

interactive_agent = ResearchAgent()

while True:
    try:
        user_input = input("\n👤 You: ").strip()
        
        if not user_input:
            continue
        if user_input.lower() == 'quit':
            print("\n👋 Goodbye!")
            break
        if user_input.lower() == 'memory':
            interactive_agent.show_memory()
            continue
        if user_input.lower() == 'clear':
            interactive_agent.clear_memory()
            interactive_agent.clear_notes()
            continue
        
        response = interactive_agent.chat(user_input, verbose=False)
        print(f"\n🤖 Agent: {response}")
        
    except KeyboardInterrupt:
        print("\n\n👋 Session ended.")
        break